# MediBot — Component 1: Document Ingestion (step-by-step)

Run the cells top to bottom. Each stage from `ingest.py` / `config.py` gets its own cell, followed by a print/inspect cell so you can verify the output before moving on.

1. Setup + config
2. Discover documents
3. Parse **one** document with Docling (structure-aware)
4. Chunk that document with HybridChunker
5. Tag metadata, embed, and upsert into Qdrant — still just the one document, to verify each piece works
6. Full ingestion loop over every remaining document
7. Verify the Qdrant collection

## 0 — Install dependencies

Installs into the **active kernel's** environment (`%pip`, not `!pip`, so it targets the kernel even if your terminal's `python3` is a different interpreter). Run this once; skip it on later runs once the packages are already installed.

In [1]:
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


## 1 — Imports

In [2]:
import logging
from pathlib import Path

from docling.document_converter import DocumentConverter
from docling.chunking import HybridChunker
from fastembed import SparseTextEmbedding
from hierarchical.postprocessor import ResultPostprocessor
from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    Modifier,
    PointStruct,
    SparseVector,
    SparseVectorParams,
    VectorParams,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("medibot.ingest")

/Users/vennilave23/Documents/VSCode_Workspace/CodeBasics/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2 — Config

Inlined from `medibot/config.py` so this notebook runs standalone from `notebooks/`.

In [3]:
def _find_rag_dir(start: Path) -> Path:
    """Walk upward from the kernel's cwd until we find the 2_RAG folder (identified by its resources subfolder)."""
    for candidate in [start, *start.parents]:
        if (candidate / "Medibot_Assignment_Resources").exists():
            return candidate
    raise FileNotFoundError("Could not locate the 2_RAG folder (looked for Medibot_Assignment_Resources/)")

RAG_ASSIGNMENT_DIR = _find_rag_dir(Path.cwd())
DATA_DIR = RAG_ASSIGNMENT_DIR / "Medibot_Assignment_Resources" / "mediassist_data"
QDRANT_PATH = str(RAG_ASSIGNMENT_DIR / "medibot" / "qdrant_storage")
COLLECTION_NAME = "medibot_docs"

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
MAX_TOKENS_PER_CHUNK = 256

# Component 2: Hybrid RAG (dense + BM25 sparse), stored as two named vectors
# on the same point so Qdrant can fuse them server-side in one query.
DENSE_VECTOR_NAME = "dense"
SPARSE_VECTOR_NAME = "sparse"
SPARSE_EMBED_MODEL = "Qdrant/bm25"

COLLECTION_ACCESS_ROLES = {
    "general": ["doctor", "nurse", "billing_executive", "technician", "admin"],
    "clinical": ["doctor", "admin"],
    "nursing": ["nurse", "doctor", "admin"],
    "billing": ["billing_executive", "admin"],
    "equipment": ["technician", "admin"],
}

SUPPORTED_SUFFIXES = {".pdf", ".md"}

print(f"RAG assignment dir: {RAG_ASSIGNMENT_DIR}")
print(f"Data dir: {DATA_DIR}")
print(f"Qdrant path: {QDRANT_PATH}")

RAG assignment dir: /Users/vennilave23/Documents/VSCode_Workspace/CodeBasics/Assignments/2_RAG
Data dir: /Users/vennilave23/Documents/VSCode_Workspace/CodeBasics/Assignments/2_RAG/Medibot_Assignment_Resources/mediassist_data
Qdrant path: /Users/vennilave23/Documents/VSCode_Workspace/CodeBasics/Assignments/2_RAG/medibot/qdrant_storage


## 3 — Discover documents

In [4]:
def discover_documents(data_dir: Path) -> list[tuple[Path, str]]:
    """Walk each role-mapped collection folder; skip db/ (that's SQL RAG data, not vector-store content)."""
    discovered = []
    for collection_dir in sorted(data_dir.iterdir()):
        if not collection_dir.is_dir() or collection_dir.name not in COLLECTION_ACCESS_ROLES:
            continue
        for file_path in sorted(collection_dir.iterdir()):
            if file_path.suffix.lower() in SUPPORTED_SUFFIXES:
                discovered.append((file_path, collection_dir.name))
    return discovered

documents = discover_documents(DATA_DIR)
print(f"Discovered {len(documents)} documents across {len(COLLECTION_ACCESS_ROLES)} collections\n")
for file_path, collection in documents:
    print(f"  [{collection}] {file_path.name}")

Discovered 12 documents across 5 collections

  [billing] billing_codes.pdf
  [billing] claim_submission_guide.md
  [clinical] diagnostic_reference.pdf
  [clinical] drug_formulary.pdf
  [clinical] treatment_protocols.pdf
  [equipment] equipment_manual.pdf
  [general] code_of_conduct.pdf
  [general] general_faqs.pdf
  [general] leave_policy.pdf
  [general] staff_handbook.pdf
  [nursing] icu_nursing_procedures.pdf
  [nursing] infection_control.pdf


## 4 — Parse one document with Docling

We test on the first discovered document before looping over all of them, so you can inspect the structure-aware output (headings/tables preserved) before committing to a full run.

In [5]:
def parse_document(file_path: Path):
    """Parse a PDF/Markdown file into a structure-aware DoclingDocument (headings/tables/code preserved).

    ResultPostprocessor infers heading hierarchy from PDF provenance (item.prov -> page/bbox), which
    Markdown-sourced items don't carry -- so it's only run for PDFs; Markdown already has explicit
    heading structure (#, ##) that HybridChunker can use directly.
    """
    converter = DocumentConverter()
    result = converter.convert(str(file_path))
    if file_path.suffix.lower() == ".pdf":
        ResultPostprocessor(result).process()
    return result.document

test_file, test_collection = documents[0]
print(f"Parsing: {test_file.name}  (collection={test_collection})")
test_dl_doc = parse_document(test_file)
print(f"Parsed OK. Document has {len(test_dl_doc.texts)} text items, {len(test_dl_doc.tables)} tables.")

2026-09-07 09:13:19,261 INFO detected formats: [<InputFormat.PDF: 'pdf'>]


Parsing: billing_codes.pdf  (collection=billing)


2026-09-07 09:13:19,534 INFO Going to convert document batch...


2026-09-07 09:13:19,535 INFO Initializing pipeline for StandardPdfPipeline with options hash e1efaab647f0694a7fd1c6d5d6812e54


2026-09-07 09:13:19,540 INFO Loading plugin 'docling_defaults'


...

2026-09-07 09:13:20,837 INFO Accelerator device: 'mps'


2026-09-07 09:13:21,562 INFO Processing document billing_codes.pdf


2026-09-07 09:13:35,021 INFO Finished converting document billing_codes.pdf in 15.76 sec.


Path or String-sources must point to a local path that exists or to HTTP or HTTPS URLs. Got: billing_codes.pdf


Parsed OK. Document has 59 text items, 7 tables.


## 5 — Load embedders, tokenizer, chunker

Two embedders: the dense sentence-transformer (semantic similarity) and a BM25 sparse embedder (exact keyword matching, via FastEmbed). Both get stored as separate named vectors on every point — see Component 2 (`notebooks/component2_hybrid_rag.ipynb`) for why.

In [6]:
embedder = SentenceTransformer(EMBED_MODEL)
sparse_embedder = SparseTextEmbedding(model_name=SPARSE_EMBED_MODEL)
tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL)
chunker = HybridChunker(tokenizer=tokenizer, max_tokens=MAX_TOKENS_PER_CHUNK, merge_peers=True)

print(f"Dense embedder loaded: {EMBED_MODEL}")
print(f"Embedding dimension: {embedder.get_sentence_embedding_dimension()}")
print(f"Sparse embedder loaded: {SPARSE_EMBED_MODEL}")

2026-09-07 09:13:35,067 INFO No device provided, using mps


2026-09-07 09:13:35,178 INFO HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"


2026-09-07 09:13:35,179 WARNING Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


2026-09-07 09:13:35,199 INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"


2026-09-07 09:13:35,254 INFO HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"


...

2026-09-07 09:13:37,356 INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/tokenizer_config.json "HTTP/1.1 200 OK"


2026-09-07 09:13:37,414 INFO HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"


2026-09-07 09:13:37,482 INFO HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"


Dense embedder loaded: sentence-transformers/all-MiniLM-L6-v2
Embedding dimension: 384
Sparse embedder loaded: Qdrant/bm25


/var/folders/pg/y49g43_121gbwclljyf9msgc0000gn/T/ipykernel_28101/4082958875.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Embedding dimension: {embedder.get_sentence_embedding_dimension()}")


## 6 — Chunk the parsed document with HybridChunker

In [7]:
def classify_chunk_type(doc_chunk) -> str:
    """Map a chunk's source item labels to the assignment's chunk_type vocabulary: text/table/heading/code."""
    labels = [str(getattr(item, "label", "")).lower() for item in doc_chunk.meta.doc_items]
    if any("table" in label for label in labels):
        return "table"
    if any("code" in label for label in labels):
        return "code"
    if any("header" in label or "title" in label for label in labels):
        return "heading"
    return "text"


def chunk_document(dl_doc, chunker: HybridChunker) -> list[dict]:
    """Two-pass chunking: HybridChunker first splits along section/subsection/paragraph structure,
    then enforces the tokenizer-aware max_tokens budget — exactly the strategy the assignment requires.
    """
    chunks = []
    for doc_chunk in chunker.chunk(dl_doc=dl_doc):
        headings = doc_chunk.meta.headings or []
        chunks.append(
            {
                "chunk_text": chunker.serialize(chunk=doc_chunk),
                "section_title": headings[-1] if headings else "Untitled Section",
                "chunk_type": classify_chunk_type(doc_chunk),
            }
        )
    return chunks

test_chunks = chunk_document(test_dl_doc, chunker)
print(f"{test_file.name} -> {len(test_chunks)} chunks\n")
print("Sample chunk (#0):")
print(f"  section_title: {test_chunks[0]['section_title']}")
print(f"  chunk_type:    {test_chunks[0]['chunk_type']}")
print(f"  chunk_text:    {test_chunks[0]['chunk_text'][:300]}...")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1453 > 512). Running this sequence through the model will result in indexing errors


billing_codes.pdf -> 23 chunks

Sample chunk (#0):
  section_title: Insurance Billing Code Reference
  chunk_type:    text
  chunk_text:    Insurance Billing Code Reference
ICD-10 Diagnosis Codes, Procedure Codes & Insurer Package Rates
MediAssist Health Network Central Billing Office Document ref: BILL-CODE-010 · Version 8.0 Access: Billing Executives & Admin...


/var/folders/pg/y49g43_121gbwclljyf9msgc0000gn/T/ipykernel_28101/543140825.py:22: DeprecationWarning: Use contextualize() instead.
  "chunk_text": chunker.serialize(chunk=doc_chunk),


## 7 — Tag metadata (source_document, collection, access_roles)

In [8]:
access_roles = COLLECTION_ACCESS_ROLES[test_collection]
for chunk in test_chunks:
    chunk["source_document"] = test_file.name
    chunk["collection"] = test_collection
    chunk["access_roles"] = access_roles

print("Sample chunk with full metadata:")
for key, value in test_chunks[0].items():
    print(f"  {key}: {value}")

Sample chunk with full metadata:
  chunk_text: Insurance Billing Code Reference
ICD-10 Diagnosis Codes, Procedure Codes & Insurer Package Rates
MediAssist Health Network Central Billing Office Document ref: BILL-CODE-010 · Version 8.0 Access: Billing Executives & Admin
  section_title: Insurance Billing Code Reference
  chunk_type: text
  source_document: billing_codes.pdf
  collection: billing
  access_roles: ['billing_executive', 'admin']


## 8 — Embed (dense + sparse) + upsert into Qdrant (single document, to verify)

In [9]:
def build_points(
    chunks_with_meta: list[dict],
    embedder: SentenceTransformer,
    sparse_embedder: SparseTextEmbedding,
    start_id: int,
) -> list[PointStruct]:
    """Embed each chunk with both the dense model and BM25, as two named vectors on one point.

    Storing both on the same point (rather than in separate collections/queries) is what lets
    Component 2 fuse dense + sparse server-side in a single Qdrant query.
    """
    texts = [c["chunk_text"] for c in chunks_with_meta]
    dense_vectors = embedder.encode(texts, show_progress_bar=False)
    sparse_vectors = list(sparse_embedder.embed(texts))
    return [
        PointStruct(
            id=start_id + i,
            vector={
                DENSE_VECTOR_NAME: dense_vector.tolist(),
                SPARSE_VECTOR_NAME: SparseVector(
                    indices=sparse_vector.indices.tolist(), values=sparse_vector.values.tolist()
                ),
            },
            payload=chunk,
        )
        for i, (chunk, dense_vector, sparse_vector) in enumerate(zip(chunks_with_meta, dense_vectors, sparse_vectors))
    ]

test_points = build_points(test_chunks, embedder, sparse_embedder, start_id=0)
print(f"Built {len(test_points)} points.")
print(f"Dense vector length: {len(test_points[0].vector[DENSE_VECTOR_NAME])}")
print(f"Sparse vector non-zero terms: {len(test_points[0].vector[SPARSE_VECTOR_NAME].indices)}")

Built 23 points.
Dense vector length: 384
Sparse vector non-zero terms: 24


In [10]:
def ensure_hybrid_collection(client: QdrantClient, embedder: SentenceTransformer) -> None:
    """Create the collection with named dense + sparse vector configs.

    A collection's vector config is immutable after creation, so a collection built by an
    older dense-only run can't be upgraded in place -- it's dropped and rebuilt from source
    documents instead (ingestion is idempotent, so this is safe to re-run).
    """
    if client.collection_exists(collection_name=COLLECTION_NAME):
        info = client.get_collection(collection_name=COLLECTION_NAME)
        has_hybrid_schema = isinstance(info.config.params.vectors, dict) and DENSE_VECTOR_NAME in (
            info.config.params.vectors or {}
        )
        if has_hybrid_schema:
            print(f"Collection '{COLLECTION_NAME}' already exists with the hybrid schema")
            return
        print(f"Collection '{COLLECTION_NAME}' exists with an old (dense-only) schema; recreating for hybrid search.")
        client.delete_collection(collection_name=COLLECTION_NAME)

    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config={
            DENSE_VECTOR_NAME: VectorParams(
                size=embedder.get_sentence_embedding_dimension(), distance=Distance.COSINE
            ),
        },
        sparse_vectors_config={
            # IDF modifier: FastEmbed's BM25 stores raw term-frequency weights at index time;
            # Qdrant applies the IDF term (from corpus-wide stats) at query time using this flag.
            SPARSE_VECTOR_NAME: SparseVectorParams(modifier=Modifier.IDF),
        },
    )
    print(f"Created Qdrant collection '{COLLECTION_NAME}' with dense+sparse vectors")


# Safe to re-run: local Qdrant only allows one open handle to the storage folder, even within
# the same kernel, so close any client left over from a previous run of this cell first.
if "client" in globals():
    client.close()

client = QdrantClient(path=QDRANT_PATH)
ensure_hybrid_collection(client, embedder)

Collection 'medibot_docs' already exists with the hybrid schema


In [11]:
client.upsert(collection_name=COLLECTION_NAME, points=test_points, wait=True)
print(f"Upserted {len(test_points)} points from {test_file.name}")
print(f"Collection point count: {client.count(collection_name=COLLECTION_NAME).count}")

Upserted 23 points from billing_codes.pdf
Collection point count: 252


## 9 — Full ingestion loop over every remaining document

Every stage above has now been verified on one document. Repeat the same steps for the rest of the corpus.

In [12]:
if not documents:
    print("No documents were discovered; nothing to ingest.")
elif len(documents) == 1:
    print("Only one document was indexed; no remaining documents to ingest.")
else:
    next_id = len(test_points) if "test_points" in globals() else 0
    total_chunks = next_id

    for file_path, collection in documents[1:]:
        if collection not in COLLECTION_ACCESS_ROLES:
            logger.warning("Skipping unrecognized collection '%s' for %s", collection, file_path.name)
            continue

        logger.info("Parsing %s (collection=%s)", file_path.name, collection)

        try:
            dl_doc = parse_document(file_path)
        except Exception:
            logger.exception("Failed to parse %s; skipping this file.", file_path.name)
            continue

        raw_chunks = chunk_document(dl_doc, chunker)

        if not raw_chunks:
            logger.warning("No chunks produced for %s; skipping this document.", file_path.name)
            continue

        access_roles = COLLECTION_ACCESS_ROLES[collection]
        for chunk in raw_chunks:
            chunk["source_document"] = file_path.name
            chunk["collection"] = collection
            chunk["access_roles"] = access_roles

        points = build_points(raw_chunks, embedder, sparse_embedder, start_id=next_id)
        client.upsert(collection_name=COLLECTION_NAME, points=points, wait=True)

        next_id += len(points)
        total_chunks += len(points)
        print(f"  -> {file_path.name}: {len(points)} chunks indexed")

    print(f"\nIngestion complete: {total_chunks} chunks from {len(documents)} documents indexed into '{COLLECTION_NAME}'")

2026-09-07 09:13:38,179 INFO Parsing claim_submission_guide.md (collection=billing)


2026-09-07 09:13:38,250 INFO detected formats: [<InputFormat.MD: 'md'>]


2026-09-07 09:13:38,251 INFO Going to convert document batch...


2026-09-07 09:13:38,251 INFO Initializing pipeline for SimplePipeline with options hash c51d76e87a563ae9551305f3279088c5


2026-09-07 09:13:38,252 INFO Processing document claim_submission_guide.md


...

2026-09-07 09:15:09,047 INFO Finished converting document infection_control.pdf in 6.63 sec.


Path or String-sources must point to a local path that exists or to HTTP or HTTPS URLs. Got: infection_control.pdf


/var/folders/pg/y49g43_121gbwclljyf9msgc0000gn/T/ipykernel_28101/543140825.py:22: DeprecationWarning: Use contextualize() instead.
  "chunk_text": chunker.serialize(chunk=doc_chunk),


falling back to cluster-based


  -> infection_control.pdf: 14 chunks indexed

Ingestion complete: 252 chunks from 12 documents indexed into 'medibot_docs'


## 10 — Verify the collection

In [13]:
info = client.count(collection_name=COLLECTION_NAME)
print(f"Total points in '{COLLECTION_NAME}': {info.count}")

sample, _ = client.scroll(collection_name=COLLECTION_NAME, limit=3, with_payload=True, with_vectors=False)
for point in sample:
    print(f"\nid={point.id}")
    for key, value in point.payload.items():
        preview = value if key != "chunk_text" else str(value)[:150] + "..."
        print(f"  {key}: {preview}")

Total points in 'medibot_docs': 252

id=0
  chunk_text: Insurance Billing Code Reference
ICD-10 Diagnosis Codes, Procedure Codes & Insurer Package Rates
MediAssist Health Network Central Billing Office Docu...
  section_title: Insurance Billing Code Reference
  chunk_type: text
  source_document: billing_codes.pdf
  collection: billing
  access_roles: ['billing_executive', 'admin']

id=1
  chunk_text: Introduction
This reference maps the diagnosis and procedure codes used in MediAssist claim submissions to insurer package rates. Accurate coding is t...
  section_title: Introduction
  chunk_type: text
  source_document: billing_codes.pdf
  collection: billing
  access_roles: ['billing_executive', 'admin']

id=2
  chunk_text: Note
Package rates below are indicative MediAssist negotiated rates. Final settlement depends on the patient's policy, sum insured, co-pay and sub-lim...
  section_title: Note
  chunk_type: text
  source_document: billing_codes.pdf
  collection: billing
  access_rol